# Regression Metrics

Comprehensive guide to evaluating regression models beyond just fitting.

**Why metrics matter:**
- One metric alone can mislead — always use multiple
- R² and plain MSE don't account for added complexity (use adjusted R² and RMSE instead)
- Residual plots reveal bias, heteroscedasticity, and outliers that metrics miss

We cover:
1. **Scale-dependent metrics** — MAE, MSE, RMSE, MSLE (same units as target)
2. **Scale-independent metrics** — R², adjusted R² (unitless, for comparison)
3. **Diagnostic plots** — actual vs predicted, residuals, Q-Q plots (for assumption checking)

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.datasets import load_diabetes
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    mean_squared_log_error,
    median_absolute_error,
    r2_score,
    mean_absolute_percentage_error
)

## 1. Set Up Data & Fit a Model

Use the diabetes dataset — 10 features predicting a diabetes disease progression score.

In [ ]:
X, y = load_diabetes(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=2)

model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print(f"Training set size: {X_train.shape[0]}")
print(f"Test set size: {X_test.shape[0]}")
print(f"Features: {X.shape[1]}")

## 2. Scale-Dependent Metrics

All in the **same units as the target** — interpretable but incomparable across datasets.

### MAE (Mean Absolute Error)

$$\text{MAE} = \frac{1}{n} \sum |y_i - \hat{y}_i|$$

- Average absolute difference between predictions and actuals
- Robust to outliers (compared to MSE)
- Easy to interpret: "on average, predictions are off by this much"

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
print(f"MAE: {mae:.2f}")
print(f"Interpretation: On average, predictions are off by {mae:.2f} units.")

### MSE (Mean Squared Error)

$$\text{MSE} = \frac{1}{n} \sum (y_i - \hat{y}_i)^2$$

- Average squared difference — penalizes large errors more heavily
- Often used in optimization (smooth, differentiable)
- Hard to interpret directly (units are squared)

In [ ]:
mse = mean_squared_error(y_test, y_pred)
print(f"MSE: {mse:.2f}")
print(f"Rarely used directly for evaluation — use RMSE or MAE instead.")

### RMSE (Root Mean Squared Error)

$$\text{RMSE} = \sqrt{\text{MSE}}$$

- Back in the same units as the target
- Penalizes large errors (like MSE) but interpretable
- Most common metric in practice

In [ ]:
rmse = np.sqrt(mse)
print(f"RMSE: {rmse:.2f}")
print(f"Interpretation: Typical prediction error is about {rmse:.2f} units (penalizes large errors).")

### Median Absolute Error

- Median (not mean) of absolute errors
- Even more robust to outliers than MAE
- Use when you have extreme outliers

In [ ]:
median_ae = median_absolute_error(y_test, y_pred)
print(f"Median Absolute Error: {median_ae:.2f}")

### MAPE (Mean Absolute Percentage Error)

$$\text{MAPE} = \frac{100\%}{n} \sum \left| \frac{y_i - \hat{y}_i}{y_i} \right|$$

- Percentage error — scale-independent but fails when actual values are near 0
- Good for business metrics ("model is off by X% on average")
- Avoid if target has values close to zero

In [ ]:
mape = mean_absolute_percentage_error(y_test, y_pred)
print(f"MAPE: {mape:.2f}%")

### MSLE (Mean Squared Log Error)

$$\text{MSLE} = \frac{1}{n} \sum (\log(1 + y_i) - \log(1 + \hat{y}_i))^2$$

- Metric for target values that are always positive and vary over orders of magnitude
- Penalizes underprediction more than overprediction
- Use for: prices, counts, any positive-only target

In [ ]:
# Note: MSLE requires all values >= 0 and model to predict >= 0
if (y_test >= 0).all() and (y_pred >= 0).all():
    msle = mean_squared_log_error(y_test, y_pred)
    print(f"MSLE: {msle:.4f}")
else:
    print("Target or predictions have negative values — MSLE not applicable.")

## 3. Scale-Independent Metrics

Unitless — good for comparing models on different datasets.

### R² (Coefficient of Determination)

$$R^2 = 1 - \frac{\text{SS}_{\text{res}}}{\text{SS}_{\text{tot}}} = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$

- Fraction of variance in `y` explained by the model
- 1.0 = perfect, 0 = baseline (predicting mean), negative = worse than baseline
- **Problem:** increases (or stays same) when adding any feature, even useless ones

In [ ]:
r2 = r2_score(y_test, y_pred)
print(f"R²: {r2:.4f}")
print(f"Interpretation: The model explains {r2*100:.1f}% of variance in the target.")

### Adjusted R²

$$\text{Adjusted } R^2 = 1 - \frac{(1-R^2)(n-1)}{n-k-1}$$

where `n` = number of samples, `k` = number of features.

- R² penalized for each added feature
- Only increases if new feature genuinely improves fit
- Use this when comparing models with different feature counts

In [ ]:
n = X_test.shape[0]
k = X_test.shape[1]
adjusted_r2 = 1 - ((1 - r2) * (n - 1) / (n - k - 1))
print(f"Adjusted R²: {adjusted_r2:.4f}")
print(f"\nR² vs Adjusted R²:")
print(f"  R²:          {r2:.4f}")
print(f"  Adjusted R²: {adjusted_r2:.4f}")
print(f"  Difference:  {(r2 - adjusted_r2):.4f} (penalty for {k} features)")

## 4. Summary Table

All metrics together for this model:

In [ ]:
metrics_dict = {
    'Metric': ['MAE', 'MSE', 'RMSE', 'Median AE', 'MAPE', 'R²', 'Adjusted R²'],
    'Value': [
        f"{mae:.4f}",
        f"{mse:.4f}",
        f"{rmse:.4f}",
        f"{median_ae:.4f}",
        f"{mape:.2f}%",
        f"{r2:.4f}",
        f"{adjusted_r2:.4f}"
    ],
    'Interpretation': [
        'Avg prediction error (robust)',
        'Avg squared error (rarely used)',
        'Typical prediction error (penalizes large errors)',
        'Median error (very robust)',
        'Percentage error',
        'Variance explained (0-1)',
        'Variance explained, penalized for features'
    ]
}

metrics_df = pd.DataFrame(metrics_dict)
print(metrics_df.to_string(index=False))

## 5. Residual Analysis

Residuals = actual - predicted. Plots reveal assumptions violations.

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# 1. Actual vs Predicted
axes[0, 0].scatter(y_test, y_pred, alpha=0.6)
axes[0, 0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'r--', lw=2)
axes[0, 0].set_xlabel('Actual')
axes[0, 0].set_ylabel('Predicted')
axes[0, 0].set_title('Actual vs Predicted')
axes[0, 0].grid(True, alpha=0.3)

# 2. Residuals vs Predicted (check heteroscedasticity)
axes[0, 1].scatter(y_pred, residuals, alpha=0.6)
axes[0, 1].axhline(y=0, color='r', linestyle='--', lw=2)
axes[0, 1].set_xlabel('Predicted')
axes[0, 1].set_ylabel('Residual')
axes[0, 1].set_title('Residuals vs Predicted (check for patterns)')
axes[0, 1].grid(True, alpha=0.3)

# 3. Histogram of residuals
axes[1, 0].hist(residuals, bins=15, edgecolor='black', alpha=0.7)
axes[1, 0].set_xlabel('Residual')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_title('Distribution of Residuals')
axes[1, 0].axvline(x=0, color='r', linestyle='--', lw=2)

# 4. Q-Q plot (check normality)
stats.probplot(residuals, dist="norm", plot=axes[1, 1])
axes[1, 1].set_title('Q-Q Plot (check normality)')

plt.tight_layout()
plt.show()

### What to Look For in Residual Plots:

1. **Actual vs Predicted:** Points should hug the diagonal. Far from diagonal = systematic underfitting.
2. **Residuals vs Predicted:** Should be a random cloud centered at 0. Patterns indicate:
   - Curved pattern → non-linear relationship (need polynomial features)
   - Funnel shape → heteroscedasticity (variance not constant)
3. **Histogram of Residuals:** Should look roughly normal (bell curve). Long tails = outliers.
4. **Q-Q Plot:** Points should hug the diagonal line. Deviations at ends = non-normality.

In [ ]:
print(f"Residual Statistics:")
print(f"  Mean:  {residuals.mean():.4f} (should be close to 0)")
print(f"  Std:   {residuals.std():.4f}")
print(f"  Min:   {residuals.min():.4f}")
print(f"  Max:   {residuals.max():.4f}")

## 6. When to Use Which Metric

| Scenario | Metric(s) | Why |
|---|---|---|
| General regression | RMSE + R² | Industry standard; RMSE in same units, R² interpretable |
| Outliers present | MAE + Median AE | Both robust; MAE easier to interpret |
| Comparing models (different datasets) | R² or Adjusted R² | Unitless, scale-independent |
| Adding features to model | Adjusted R² | Penalizes complexity |
| Business reporting | MAPE or MAE | Easy to explain to non-technical stakeholders |
| Positive targets (prices, counts) | MSLE or MAPE | Penalizes underprediction fairly |
| Budget/resource constraints | MAE | Constant penalty per unit error |
| Safety-critical (e.g., medical) | RMSE + Residual plots | Penalizes large errors; visual inspection |

**Best practice:** Always use multiple metrics + residual plots.

---
## Regression Metrics Cheat Sheet

### Scale-Dependent (same units as target)
| Metric | Code | When to Use |
|---|---|---|
| MAE | `mean_absolute_error(y, y_pred)` | Baseline, robust |
| RMSE | `np.sqrt(mean_squared_error(y, y_pred))` | Industry standard |
| Median AE | `median_absolute_error(y, y_pred)` | Extreme outliers |
| MAPE | `mean_absolute_percentage_error(y, y_pred)` | Business reporting |

### Scale-Independent (unitless)
| Metric | Code | When to Use |
|---|---|---|
| R² | `r2_score(y, y_pred)` | Default |
| Adjusted R² | `1 - ((1-r2)*(n-1)/(n-k-1))` | Comparing models with different features |

### Diagnostic Plots
- Actual vs Predicted: Points near diagonal = good fit
- Residuals vs Predicted: Random cloud = no patterns
- Histogram of Residuals: Bell curve = normal errors
- Q-Q Plot: Points on diagonal = normally distributed errors